In [1]:
import numpy as np

# Manual Naive Bayes computation
print("=" * 50)
print("MANUAL NAIVE BAYES — SPAM DETECTION")
print("=" * 50)

# Priors
p_spam = 0.5
p_not_spam = 0.5

# Likelihoods (with Laplace smoothing)
# P(feature=1 | class)
likelihoods_spam = {'free': 0.6, 'money': 0.6, 'meeting': 0.2}
likelihoods_not = {'free': 0.4, 'money': 0.2, 'meeting': 0.6}

# New email: free=1, money=1, meeting=0
email = {'free': 1, 'money': 1, 'meeting': 0}

# Compute unnormalized posteriors
log_p_spam = np.log(p_spam)
log_p_not = np.log(p_not_spam)

for feature, present in email.items():
    if present:
        log_p_spam += np.log(likelihoods_spam[feature])
        log_p_not += np.log(likelihoods_not[feature])
    else:
        log_p_spam += np.log(1 - likelihoods_spam[feature])
        log_p_not += np.log(1 - likelihoods_not[feature])

# Normalize (in log space)
max_log = max(log_p_spam, log_p_not)
p_spam_given_x = np.exp(log_p_spam - max_log) / (np.exp(log_p_spam - max_log) + np.exp(log_p_not - max_log))

print(f"P(spam | email) = {p_spam_given_x:.4f} = {p_spam_given_x*100:.1f}%")
print(f"Prediction: {'SPAM' if p_spam_given_x > 0.5 else 'NOT SPAM'}")

MANUAL NAIVE BAYES — SPAM DETECTION
P(spam | email) = 0.9000 = 90.0%
Prediction: SPAM


In [2]:
import numpy as np

class GaussianNBFromScratch:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.n_classes = len(self.classes)
        self.n_features = X.shape[1]
        
        # Compute mean, variance, and prior for each class
        self.means = np.zeros((self.n_classes, self.n_features))
        self.vars = np.zeros((self.n_classes, self.n_features))
        self.priors = np.zeros(self.n_classes)
        
        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.means[i] = X_c.mean(axis=0)
            self.vars[i] = X_c.var(axis=0) + 1e-9  # Add small value for stability
            self.priors[i] = len(X_c) / len(X)
        
        return self
    
    def _log_gaussian(self, x, mean, var):
        """Log of Gaussian PDF."""
        return -0.5 * np.log(2 * np.pi * var) - 0.5 * ((x - mean) ** 2) / var
    
    def predict(self, X):
        predictions = []
        for x in X:
            posteriors = []
            for i in range(self.n_classes):
                log_prior = np.log(self.priors[i])
                log_likelihood = np.sum(self._log_gaussian(x, self.means[i], self.vars[i]))
                posteriors.append(log_prior + log_likelihood)
            predictions.append(self.classes[np.argmax(posteriors)])
        return np.array(predictions)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

# Test
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)

nb = GaussianNBFromScratch()
nb.fit(X_train, y_train)
print(f"From-scratch accuracy: {nb.score(X_test, y_test):.2%}")

# Compare with sklearn
from sklearn.naive_bayes import GaussianNB
sklearn_nb = GaussianNB()
sklearn_nb.fit(X_train, y_train)
print(f"sklearn accuracy:      {sklearn_nb.score(X_test, y_test):.2%}")

From-scratch accuracy: 100.00%
sklearn accuracy:      100.00%


In [3]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Gaussian NB
gnb = GaussianNB()
gnb.fit(X_train, y_train)
print(f"Gaussian NB accuracy: {gnb.score(X_test, y_test):.4f}")
print(classification_report(y_test, gnb.predict(X_test), target_names=data.target_names))

Gaussian NB accuracy: 0.9737
              precision    recall  f1-score   support

   malignant       1.00      0.93      0.96        43
      benign       0.96      1.00      0.98        71

    accuracy                           0.97       114
   macro avg       0.98      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [4]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import numpy as np

# Sample text data
texts = [
    "free money now click here", "earn cash fast easy money",
    "buy cheap products discount", "limited offer free gift",
    "meeting tomorrow at 3pm", "project deadline next week",
    "lunch with team today", "quarterly report due friday",
    "budget review scheduled", "conference call at noon",
    "win big prize lottery", "cheap viagra pills now",
    "team building event", "code review complete",
]
labels = [1,1,1,1, 0,0,0,0, 0,0, 1,1, 0,0]  # 1=spam, 0=not spam

# Build pipeline
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', MultinomialNB(alpha=1.0))
])

pipeline.fit(texts, labels)

# Test on new emails
test_emails = [
    "free money offer today",
    "team meeting at 2pm",
    "win lottery cash prize"
]

for email in test_emails:
    pred = pipeline.predict([email])[0]
    proba = pipeline.predict_proba([email])[0]
    label = 'SPAM' if pred == 1 else 'HAM'
    print(f'"{email}" → {label} (confidence: {max(proba):.1%})')


"free money offer today" → SPAM (confidence: 59.0%)
"team meeting at 2pm" → HAM (confidence: 76.8%)
"win lottery cash prize" → SPAM (confidence: 64.5%)
